In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import inv, det

In [ ]:
N_x = 6
N_y = 3
A = (0.5 * np.random.rand(N_y, N_x, 2) + 0.5).view(dtype=np.complex128)[:, :, 0]
A.shape
A_dagger = np.conj(A).T
A, A_dagger
AAdagger = A @ A_dagger
print(AAdagger.shape)
AAdagger_inv = inv(AAdagger)
AAdagger_det = det(AAdagger)
print(AAdagger_det, AAdagger_inv.shape)
# plt.imshow(np.abs(A))
# plt.show()
# plt.imshow(np.abs(A_dagger))
# plt.show()
# plt.imshow(np.abs(AAdagger))
# plt.show()
# plt.imshow(np.abs(AAdagger_inv))

In [ ]:
x = np.random.rand(N_x, 1, 2).view(dtype=np.complex128)[:, :, 0]
y = A @ x
print(x, y)

In [ ]:
def J(x, delta_x):
    y = A @ x
    delta_y = A @ delta_x
    mu = np.abs(y) ** 2
    return 4 * A_dagger @ (1.0 / mu * np.real(np.conj(y) * delta_y) * y)


def J_plus(x, delta_x):
    y = A @ x
    delta_y = AAdagger_inv @ A @ delta_x
    mu = np.abs(y) ** 2
    return (
        1 / 4 * A_dagger @ AAdagger_inv @ (1.0 / mu * np.real(np.conj(y) * delta_y) * y)
    )


def JJ_plus(x, delta_x):
    y = A @ x
    delta_y = AAdagger_inv @ A @ delta_x
    mu = np.abs(y) ** 2
    return A_dagger @ (1.0 / mu * np.real(np.conj(y) * delta_y) * y)


def J_plusJ(x, delta_x):
    y = A @ x
    delta_y = A @ delta_x
    mu = np.abs(y) ** 2
    return A_dagger @ AAdagger_inv @ (1.0 / mu * np.real(np.conj(y) * delta_y) * y)


def ip(a, b):
    return np.real(np.conj(a).T @ b)


def I(x, delta_x):
    Jx = J(x, delta_x)
    return ip(delta_x, Jx)


def I_plus(x, delta_x):
    Jx = J_plus(x, delta_x)
    return ip(delta_x, Jx)


delta_x = 0.001 * np.random.rand(N_x, 1, 2).view(dtype=np.complex128)[:, :, 0]
I(x, x), I_plus(x, x), J(x, J_plus(x, delta_x)), JJ_plus(x, delta_x), J_plus(
    x, J(x, delta_x)
), J_plusJ(x, delta_x), ip(J(x, delta_x), x), ip(x, J(x, delta_x))

In [ ]:
def J(delta_x):
    y = A @ x
    delta_y = A @ delta_x
    mu = np.abs(y) ** 2
    return 4 * A_dagger @ (1.0 / mu * np.real(np.conj(y) * delta_y) * y)


def J_plus(delta_x):
    y = A @ x
    delta_y = AAdagger_inv @ A @ delta_x
    mu = np.abs(y) ** 2
    return (
        1 / 4 * A_dagger @ AAdagger_inv @ (1.0 / mu * np.real(np.conj(y) * delta_y) * y)
    )


J_check = np.allclose(J(delta_x), J(J_plus(J(delta_x))))
Jplus_check = np.allclose(J_plus(delta_x), J_plus(J(J_plus(delta_x))))
proj_check = np.allclose(J(J_plus(delta_x)), J_plus(J(delta_x)))
J_plus_self_adj_check = np.allclose(
    ip(delta_x, J_plus(delta_x)) / ip(delta_x, J_plus(delta_x)),
    ip(J_plus(delta_x), delta_x) / ip(delta_x, J_plus(delta_x)),
)
J_self_adj_check = np.allclose(
    ip(delta_x, J(delta_x)) / ip(delta_x, J(delta_x)),
    ip(J(delta_x), delta_x) / ip(delta_x, J(delta_x)),
)
J_self_adj_check = np.allclose(
    ip(delta_x, J(delta_x)) / ip(delta_x, J(delta_x)),
    ip(J(delta_x), delta_x) / ip(delta_x, J(delta_x)),
)
if J_check:
    print("J = J J^+ J")
if Jplus_check:
    print("J^+ = J^+ J J^+")
if J_plus_self_adj_check:
    print("J^+ self-adjoint")
if J_self_adj_check:
    print("J self-adjoint")
if proj_check:
    print("JJ^+ = J^+J")
else:
    print("JJ^+ /= J^+J")